# Pull Copernicus DEM GLO-30

In [1]:
# Add project root to Python path for imports
import sys
from pathlib import Path

# Get project root (parent of 'dem' folder)
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import functions from our geogfm module
from functions.geogfm.c01 import (
    verify_environment,
    setup_planetary_computer_auth,
    search_sentinel2_scenes,
    load_sentinel2_bands
)

# Core libraries
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.merge import merge
import rioxarray
from pathlib import Path
from datetime import datetime, timedelta
from pystac_client import Client
import folium
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from functools import partial
import dask
from dask.distributed import Client as DaskClient
from typing import Dict, List, Tuple, Optional, Union
import logging

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Verify environment using our standardized function
required_packages = [
    'numpy', 'pandas', 'xarray', 'rasterio', 'rioxarray',
    'pystac_client', 'folium', 'matplotlib', 'dask'
]
env_status = verify_environment(required_packages)

# Set up study area -Dangermond Preserve
preserve_bbox = [-120.5130681, 34.4344052,  -120.3456914, 34.5775648]  # [west, south, east, north]
buffer_deg = 0.05

# buffer bbox to accomodate patches centered around sensors near edges of preserve
preserve_bbox = [
    preserve_bbox[0] - buffer_deg,  # west
    preserve_bbox[1] - buffer_deg,  # south  
    preserve_bbox[2] + buffer_deg,  # east
    preserve_bbox[3] + buffer_deg   # north
]


# Define longer time range for trend analysis
start_date = "2023-01-01"
end_date = "2025-01-01"
max_cloud_cover = 15  # More restrictive for cleaner mosaics

print(f"Study Area: Santa Barbara, California")
print(f"Time Range: {start_date} to {end_date}")
print(f"Max Cloud Cover: {max_cloud_cover}%")

2025-11-29 14:07:22,777 - INFO - Analysis results exported to: c:\Users\danwillett\Code\geofms\dem\week1_output
2025-11-29 14:07:22,778 - INFO - Data exported - use load_week1_data() to reload
2025-11-29 14:07:23,408 - INFO - All 9 packages verified


Study Area: Santa Barbara, California
Time Range: 2023-01-01 to 2025-01-01
Max Cloud Cover: 15%


In [2]:
import planetary_computer as pc

def search_dem_scenes(
    bbox: List[float],
    limit: int = 10
) -> List:
    """
    Search Copernicus DEM 30 scenes using STAC API.

    Parameters
    ----------
    bbox : List[float]
        Bounding box as [west, south, east, north] in WGS84
    limit : int
        Maximum scenes to return

    Returns
    -------
    List[pystac.Item]
        List of STAC items sorted by cloud cover (ascending)
    """
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace
    )

    search_params = {
        "collections": ["cop-dem-glo-30"],
        "bbox": bbox,
        "limit": limit
    }

    search_results = catalog.search(**search_params)
    items = list(search_results.items())

    logger.info(f"Found {len(items)} Copernicus DEM scenes")
    return items

In [3]:
# Set up authentication using our standardized function
auth_status = setup_planetary_computer_auth()

# Search for scenes using our enhanced search function
print("Searching for multiple copernicus dem scenes...")
items = search_dem_scenes(
    bbox=preserve_bbox,
    limit=50
)

print(f"Found {len(items)} scenes")

# Organize scenes by date and tile
scene_info = []
for item in items:
    print(item)
    props = item.properties
    print(props)
    date = props['datetime'].split('T')[0]
    tile_id = item.id.split('_')[5]  # Extract tile ID from scene name

    scene_info.append({
        'id': item.id,
        'date': date,
        'tile': tile_id,
        'item': item
    })

# Convert to DataFrame for easier analysis
scenes_df = pd.DataFrame(scene_info)



2025-11-29 14:07:23,423 - INFO - Using anonymous access (basic rate limits)


Searching for multiple copernicus dem scenes...


2025-11-29 14:07:25,337 - INFO - Found 1 Copernicus DEM scenes


Found 1 scenes
<Item id=Copernicus_DSM_COG_10_N34_00_W121_00_DEM>
{'gsd': 30, 'datetime': '2021-04-22T00:00:00Z', 'platform': 'TanDEM-X', 'proj:shape': [3600, 3600], 'proj:transform': [0.0002777777777777778, 0.0, -121.00013888888888, 0.0, -0.0002777777777777778, 35.00013888888889], 'proj:code': 'EPSG:4326'}


In [4]:
# Create map showing all scene footprints
m = folium.Map(
    location=[34.45, -119.85],  # Center of Santa Barbara
    zoom_start=10,
    tiles='OpenStreetMap'
)

# Add study area boundary
folium.Rectangle(
    bounds=[[preserve_bbox[1], preserve_bbox[0]],
            [preserve_bbox[3], preserve_bbox[2]]],
    color='red',
    fill=False,
    weight=3,
    popup="Study Area: Dangermond Preserve"
).add_to(m)

# Add scene footprints colored by date
colors = ['blue', 'green', 'orange', 'purple', 'red']
unique_dates = sorted(scenes_df['date'].unique())

for i, date in enumerate(unique_dates[:5]):  # Show first 5 dates
    date_scenes = scenes_df[scenes_df['date'] == date]
    color = colors[i % len(colors)]

    for _, scene in date_scenes.iterrows():
        item = scene['item']
        geom = item.geometry

        # Add scene footprint
        folium.GeoJson(
            geom,
            style_function=lambda x, color=color: {
                'fillColor': color,
                'color': color,
                'weight': 2,
                'fillOpacity': 0.3
            },
            popup=f"Date: {date}<br>Tile: {scene['tile']}"
        ).add_to(m)

folium.LayerControl().add_to(m)
print("Scene coverage map created")
m

Scene coverage map created


In [5]:
import rioxarray as rxr
import numpy as np
from pyproj import Transformer

# Download DEM tile (COG format - Cloud Optimized GeoTIFF)
item = items[0]  # Your tile
dem_url = item.assets['data'].href

print(f"Downloading DEM from: {dem_url[:50]}...")
dem = rxr.open_rasterio(dem_url, masked=True)

print(f"\n📊 Original DEM:")
print(f"  Shape: {dem.shape}")
print(f"  Resolution: {dem.rio.resolution()} (degrees)")
print(f"  CRS: {dem.rio.crs}")
print(f"  Bounds: {dem.rio.bounds()}")

# Clip to preserve bounding box

dem_preserve = dem.rio.clip_box(
    minx=preserve_bbox[0],
    miny=preserve_bbox[1],
    maxx=preserve_bbox[2],
    maxy=preserve_bbox[3]
)

print(f"\n📊 Clipped DEM:")
print(f"  Shape: {dem_preserve.shape}")
print(f"  Data range: [{dem_preserve.min().values:.1f}, {dem_preserve.max().values:.1f}] meters")

# Reproject from WGS84 to UTM Zone 10N (to match radar)
target_crs = 'EPSG:32610'  # UTM Zone 10N

dem_utm = dem_preserve.rio.reproject(target_crs)

print(f"\n📊 Reprojected DEM:")
print(f"  CRS: {dem_utm.rio.crs}")
print(f"  Resolution: {dem_utm.rio.resolution()} meters")
print(f"  Bounds (UTM): {dem_utm.rio.bounds()}")


from rasterio.enums import Resampling

# Resample from 30m to 10m
scale_factor = 3  # 30m → 10m

dem_10m = dem_utm.rio.reproject(
    dem_utm.rio.crs,
    resolution=(10, 10),  # Target: 10m × 10m pixels
    resampling=Resampling.bilinear  # Smooth interpolation
)

print(f"\n📊 Resampled DEM (10m):")
print(f"  Shape: {dem_10m.shape}")
print(f"  Resolution: {dem_10m.rio.resolution()} meters")
print(f"  Coverage: {dem_10m.shape[1] * 10}m × {dem_10m.shape[2] * 10}m")

# Save final DEM
dem_10m.rio.to_raster('../preserve_dem_10m_utm.tif')
print("\n✓ Saved: preserve_dem_10m_utm.tif (ready for training!)")


📊 Original DEM:
  Shape: (1, 3600, 3600)
  Resolution: (0.0002777777777777778, -0.0002777777777777778) (degrees)
  CRS: EPSG:4326
  Bounds: (-121.00013888888888, 34.000138888888884, -120.0001388888889, 35.00013888888889)

📊 Clipped DEM:
  Shape: (1, 876, 964)
  Data range: [0.0, 652.5] meters

📊 Reprojected DEM:
  CRS: EPSG:32610
  Resolution: (28.041318382575106, -28.041318382575106) meters
  Bounds (UTM): (723389.2029239134, 3807456.982624998, 748654.4307866136, 3835077.681231834)

📊 Resampled DEM (10m):
  Shape: (1, 2763, 2527)
  Resolution: (10.0, -10.0) meters
  Coverage: 27630m × 25270m

✓ Saved: preserve_dem_10m_utm.tif (ready for training!)
